In [1]:
import pandas as pd
import numpy as np

import nltk
import string
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
print('done')

done


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/shohag-
[nltk_data]     rana/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/shohag-
[nltk_data]     rana/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("Emotion_Sentiment_DataSet.csv")

In [4]:
df.head(10)

,Unnamed: 0,Text,Emotion
0,0,i feel jealous becasue i wanted that kind of l...,love
1,1,i feel like she has taken on the role of a gra...,love
2,2,i feel like im back to the arms of a beloved l...,love
3,3,i am feeling so festive right now and not just...,love
4,4,i don t discuss even my feelings for beloved w...,love
5,5,i also love seeing a star emerge and i feel li...,love
6,6,im feeling kinda shaky my mind is full of doub...,love
7,7,i love running because i feel strong and power...,love
8,8,i feel that popular bloggers dont post with fr...,love
9,9,i can never fall in love with anyone because m...,love


In [5]:
df.shape

(160000, 3)

In [6]:
df.columns

Index(['Unnamed: 0', 'Text', 'Emotion'], dtype='str')

In [7]:
df.isnull().sum()

Unnamed: 0    0
Text          8
Emotion       0
dtype: int64

In [8]:
df["Emotion"].value_counts()

Emotion
love          39553
happiness     27175
sadness       17481
Normal        16351
hate          15267
anger         12336
Depression    10333
fun           10075
surprise       6954
worry          4475
Name: count, dtype: int64

In [9]:
df=df.drop(columns=["Unnamed: 0"])

In [10]:
df.head()

,Text,Emotion
0,i feel jealous becasue i wanted that kind of l...,love
1,i feel like she has taken on the role of a gra...,love
2,i feel like im back to the arms of a beloved l...,love
3,i am feeling so festive right now and not just...,love
4,i don t discuss even my feelings for beloved w...,love


In [11]:
nlp = spacy.blank("en")  

In [12]:
texts = df["Text"].fillna("").astype(str)

docs = nlp.pipe(texts, batch_size=512)

df["tokens"] = [[token.text for token in doc] for doc in docs]

In [13]:


df.to_pickle("df_tokenized.pkl")

In [14]:
df["tokens"].head(10)

0    [i, feel, jealous, becasue, i, wanted, that, k...
1    [i, feel, like, she, has, taken, on, the, role...
2    [i, feel, like, i, m, back, to, the, arms, of,...
3    [i, am, feeling, so, festive, right, now, and,...
4    [i, don, t, discuss, even, my, feelings, for, ...
5    [i, also, love, seeing, a, star, emerge, and, ...
6    [i, m, feeling, kinda, shaky, my, mind, is, fu...
7    [i, love, running, because, i, feel, strong, a...
8    [i, feel, that, popular, bloggers, do, nt, pos...
9    [i, can, never, fall, in, love, with, anyone, ...
Name: tokens, dtype: object

In [15]:
df["tokens"] = df["tokens"].apply(lambda x: [word.lower() for word in x])

In [16]:
df.to_pickle("step2_case_folding.pkl")

In [17]:

stop_words = set(stopwords.words("english"))

df["tokens"] = df["tokens"].apply(
    lambda x: [word for word in x if word not in stop_words]
)

In [18]:
df.to_pickle("step3_stop_word_removed.pkl")

In [19]:
stemmer = PorterStemmer()

df["tokens"] = df["tokens"].apply(
    lambda x: [stemmer.stem(word) for word in x]
)

In [20]:
df.to_pickle("step4_stemmer_removed.pkl")

In [21]:
nlp = spacy.load("en_core_web_sm")

df["lemmas"] = df["Text"].fillna("").apply(
    lambda text: [token.lemma_ for token in nlp(str(text))]
)

In [22]:
df.to_pickle("step5_stemmer_removed.pkl")

In [23]:
df["processed_text"] = df["lemmas"].apply(" ".join)

In [24]:
df.to_pickle("step6_process_text.pkl")

In [25]:
df.head()

,Text,Emotion,tokens,lemmas,processed_text
0,i feel jealous becasue i wanted that kind of l...,love,"[feel, jealou, becasu, want, kind, love, true,...","[I, feel, jealous, becasue, I, want, that, kin...",I feel jealous becasue I want that kind of lov...
1,i feel like she has taken on the role of a gra...,love,"[feel, like, taken, role, grandmoth, sinc, bel...","[I, feel, like, she, have, take, on, the, role...",I feel like she have take on the role of a gra...
2,i feel like im back to the arms of a beloved l...,love,"[feel, like, back, arm, belov, last, seen, lon...","[I, feel, like, I, m, back, to, the, arm, of, ...",I feel like I m back to the arm of a beloved l...
3,i am feeling so festive right now and not just...,love,"[feel, festiv, right, love, wintri, scene, wal...","[I, be, feel, so, festive, right, now, and, no...",I be feel so festive right now and not just be...
4,i don t discuss even my feelings for beloved w...,love,"[discuss, even, feel, belov, anyon]","[I, don, t, discuss, even, my, feeling, for, b...",I don t discuss even my feeling for beloved wi...


In [26]:
X = df["processed_text"]
y = df["Emotion"]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [28]:
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [29]:

nb = MultinomialNB()

nb.fit(X_train_bow, y_train)

y_pred = nb.predict(X_test_bow)

In [30]:

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred, average="weighted"))

Accuracy : 0.86146875
Precision: 0.8908691434209736
Recall   : 0.86146875
F1 Score : 0.8600100409535867


In [31]:
print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

[[2000   11    1    0    7   12   12   11    1    1]
 [ 607 1809   19   23  148   46  581   55    7    6]
 [  91    5 2101    0   68   17  179   60    2    0]
 [ 115    4    5 1567   19    5  301   28    2    0]
 [ 104    3    3   11 5037   12  200   27    0    0]
 [ 109    3    3    4    9 2675  206   18    0    0]
 [ 114    6    7    3   30   13 7529   29    2    0]
 [ 244    3    1    4   23    4   33 3269    0    0]
 [  85    0    3    4   19   11   56   80 1143    0]
 [ 200    7    8    3   62    6  122   89    1  437]]
              precision    recall  f1-score   support

  Depression       0.55      0.97      0.70      2056
      Normal       0.98      0.55      0.70      3301
       anger       0.98      0.83      0.90      2523
         fun       0.97      0.77      0.86      2046
   happiness       0.93      0.93      0.93      5397
        hate       0.96      0.88      0.92      3027
        love       0.82      0.97      0.89      7733
     sadness       0.89      0.91   

In [32]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_bow, y_train)

y_pred_lr_bow = lr.predict(X_test_bow)

In [33]:
print("========== Logistic Regression (BoW) ==========")

print("Accuracy :", accuracy_score(y_test, y_pred_lr_bow))
print("Precision:", precision_score(y_test, y_pred_lr_bow, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_lr_bow, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_lr_bow, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_lr_bow))

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr_bow))

========== Logistic Regression (BoW) ==========
Accuracy : 0.98390625
Precision: 0.9837587683364096
Recall   : 0.98390625
F1 Score : 0.9837834960838183

Confusion Matrix
[[1858  119    5    5   12   17   10   26    0    4]
 [  61 3135    9    9   25   10   29   11    3    9]
 [   0    0 2518    0    2    2    1    0    0    0]
 [   2    6    1 2032    3    1    1    0    0    0]
 [   4    3    1    0 5371   10    8    0    0    0]
 [   7    1    2    0    6 3004    6    0    0    1]
 [   5    5    0    0   17    5 7701    0    0    0]
 [  14    0    0    3    8    2   12 3542    0    0]
 [   0    0    0    1    0    0    0    0 1400    0]
 [   2    0    2    1    3    0    3    0    0  924]]

Classification Report
              precision    recall  f1-score   support

  Depression       0.95      0.90      0.93      2056
      Normal       0.96      0.95      0.95      3301
       anger       0.99      1.00      1.00      2523
         fun       0.99      0.99      0.99      2046
   ha

In [34]:
svm = LinearSVC()

svm.fit(X_train_bow, y_train)

y_pred = svm.predict(X_test_bow)

In [35]:

print("========== Linear SVC (BoW) ==========")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

========== Linear SVC (BoW) ==========
Accuracy : 0.982125
Precision: 0.9819106505207507
Recall   : 0.982125
F1 Score : 0.9818802229047702

Confusion Matrix
[[1848   96    5    9   11   23   14   37    5    8]
 [  78 3027   16   20   34   29   48   26    9   14]
 [   0    0 2523    0    0    0    0    0    0    0]
 [   0    2    0 2041    3    0    0    0    0    0]
 [   4    2    0    0 5377    8    4    2    0    0]
 [   0    0    2    2    1 3017    4    0    0    1]
 [   3    2    0    2    7    7 7710    0    0    2]
 [  14    0    0    0    4    0    8 3555    0    0]
 [   0    0    0    0    0    0    0    0 1401    0]
 [   2    0    2    0    0    0    2    0    0  929]]

Classification Report
              precision    recall  f1-score   support

  Depression       0.95      0.90      0.92      2056
      Normal       0.97      0.92      0.94      3301
       anger       0.99      1.00      1.00      2523
         fun       0.98      1.00      0.99      2046
   happiness      

In [36]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [37]:
nb_tfidf = MultinomialNB()

nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

In [38]:
print("Accuracy :", accuracy_score(y_test, y_pred_nb_tfidf))
print("Precision:", precision_score(y_test, y_pred_nb_tfidf, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_nb_tfidf, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_nb_tfidf, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_nb_tfidf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_nb_tfidf))

Accuracy : 0.60365625
Precision: 0.7933429718002504
Recall   : 0.60365625
F1 Score : 0.5772571927518781

Confusion Matrix
[[1536    6    1    0   24    4  474   11    0    0]
 [ 121 1064    4    2  210   14 1857   29    0    0]
 [   6    0  799    0  204    4 1402  108    0    0]
 [  14    5    2  278   54    1 1676   16    0    0]
 [   1    2    0    1 4039    0 1350    4    0    0]
 [  10    3    2    0   26 1072 1908    6    0    0]
 [   2    3    0    0   29    5 7692    2    0    0]
 [  85    3    1    2   87    2  822 2579    0    0]
 [   6    9    2    1  125    2  882  133  241    0]
 [  50    1    2    0  167    3  639   56    0   17]]

Classification Report
              precision    recall  f1-score   support

  Depression       0.84      0.75      0.79      2056
      Normal       0.97      0.32      0.48      3301
       anger       0.98      0.32      0.48      2523
         fun       0.98      0.14      0.24      2046
   happiness       0.81      0.75      0.78      5397

In [39]:

lr_tfidf = LogisticRegression(max_iter=1000)

lr_tfidf.fit(X_train_tfidf, y_train)

y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

In [40]:
print("Accuracy :", accuracy_score(y_test, y_pred_lr_tfidf))
print("Precision:", precision_score(y_test, y_pred_lr_tfidf, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_lr_tfidf, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_lr_tfidf, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_lr_tfidf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr_tfidf))

Accuracy : 0.9710625
Precision: 0.9708740063897358
Recall   : 0.9710625
F1 Score : 0.9708836354892991

Confusion Matrix
[[1839  119    0    2   20   22   17   32    0    5]
 [  87 3015   12   18   43   17   67   24    5   13]
 [   3    7 2505    0    1    2    4    0    0    1]
 [   5   15    9 1992   10    4   10    0    0    1]
 [   7   14    3    0 5340    6   25    0    0    2]
 [  10    6    8    0   10 2976   16    0    0    1]
 [   4    8    3    1   29   18 7668    0    0    2]
 [  19   10    1    7   25   11   32 3472    2    2]
 [   1    8    4    2    0    4    9    0 1373    0]
 [   5    1    4    4    7    2   12    0    6  894]]

Classification Report
              precision    recall  f1-score   support

  Depression       0.93      0.89      0.91      2056
      Normal       0.94      0.91      0.93      3301
       anger       0.98      0.99      0.99      2523
         fun       0.98      0.97      0.98      2046
   happiness       0.97      0.99      0.98      5397
 

In [41]:


svm_tfidf = LinearSVC()

svm_tfidf.fit(X_train_tfidf, y_train)

y_pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)

In [42]:
print("Accuracy :", accuracy_score(y_test, y_pred_svm_tfidf))
print("Precision:", precision_score(y_test, y_pred_svm_tfidf, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_svm_tfidf, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_svm_tfidf, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_svm_tfidf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_svm_tfidf))

Accuracy : 0.980375
Precision: 0.9801996025361107
Recall   : 0.980375
F1 Score : 0.9801319850919021

Confusion Matrix
[[1870   90    2    3   16   19   11   37    1    7]
 [  73 3005   15   20   53   23   63   27    9   13]
 [   0    0 2520    0    0    2    1    0    0    0]
 [   0    0    0 2040    1    1    4    0    0    0]
 [   5    3    1    0 5368    8   10    0    0    2]
 [   0    1    8    0    8 3003    6    0    0    1]
 [   2    2    7    0   17    7 7694    0    1    3]
 [   7    0    0    2    5    3   13 3551    0    0]
 [   0    2    0    0    0    0    1    0 1398    0]
 [   0    0    2    1    3    0    5    0    1  923]]

Classification Report
              precision    recall  f1-score   support

  Depression       0.96      0.91      0.93      2056
      Normal       0.97      0.91      0.94      3301
       anger       0.99      1.00      0.99      2523
         fun       0.99      1.00      0.99      2046
   happiness       0.98      0.99      0.99      5397
   

In [ ]:
import numpy as np
import pandas as pd

def show_all_model_prediction(text):
    
    models = {
        "Naive Bayes (BOW)": (nb, bow),
        "Logistic Regression (BOW)": (lr, bow),
        "SVM (BOW)": (svm, bow),
        "Naive Bayes (TF-IDF)": (nb_tfidf, tfidf),
        "Logistic Regression (TF-IDF)": (lr_tfidf, tfidf),
        "SVM (TF-IDF)": (svm_tfidf, tfidf)
    }

    print("="*70)
    print("Input Text:")
    print(text)
    print("="*70)

    for name, (model, vectorizer) in models.items():

        x = vectorizer.transform([text])

        prediction = model.predict(x)[0]

        
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(x)[0]

        else:
           
            scores = model.decision_function(x)[0]

            exp_scores = np.exp(scores - np.max(scores))
            probs = exp_scores / exp_scores.sum()


        classes = model.classes_

        result = pd.DataFrame({
            "Class": classes,
            "Percentage": probs * 100
        })

        result = result.sort_values(
            by="Percentage",
            ascending=False
        )


        print("\nMODEL:", name)
        print("Prediction:", prediction)

        print(result.to_string(
            index=False,
            formatters={
                "Percentage": "{:.2f}%".format
            }
        ))

        print("-"*70)

In [44]:
%whos

Variable                    Type                    Data/Info
-------------------------------------------------------------
CountVectorizer             type                    <class 'sklearn.feature_e<...>on.text.CountVectorizer'>
LinearSVC                   type                    <class 'sklearn.svm._classes.LinearSVC'>
LogisticRegression          type                    <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
MultinomialNB               ABCMeta                 <class 'sklearn.naive_bayes.MultinomialNB'>
PorterStemmer               ABCMeta                 <class 'nltk.stem.porter.PorterStemmer'>
TfidfVectorizer             type                    <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
X                           Series                  Shape: (160000,)
X_test                      Series                  Shape: (32000,)
X_test_bow                  csr_matrix              <Compressed Sparse Row sp<...>06)	1\n  (31999, 40062)	1
X_test_tfidf             

In [ ]:
import numpy as np
import pandas as pd


def show_all_model_prediction(text):

    models = {
        "Naive Bayes (BOW)": (nb, bow),
        "Logistic Regression (BOW)": (lr, bow),
        "SVM (BOW)": (svm, bow),

        "Naive Bayes (TF-IDF)": (nb_tfidf, tfidf),
        "Logistic Regression (TF-IDF)": (lr_tfidf, tfidf),
        "SVM (TF-IDF)": (svm_tfidf, tfidf)
    }


    print("="*70)
    print("Input Text:")
    print(text)
    print("="*70)


    for name, (model, vectorizer) in models.items():

        try:

            x = vectorizer.transform([text])

            prediction = model.predict(x)[0]


            if hasattr(model, "predict_proba"):

                probabilities = model.predict_proba(x)[0]


            else:

                scores = model.decision_function(x)

               
                if len(scores.shape) > 1:
                    scores = scores[0]


                exp_scores = np.exp(
                    scores - np.max(scores)
                )

                probabilities = exp_scores / exp_scores.sum()



            classes = model.classes_


            result = pd.DataFrame({
                "Class": classes,
                "Confidence": probabilities * 100
            })


            result = result.sort_values(
                by="Confidence",
                ascending=False
            )


            print("\nMODEL:", name)
            print("Prediction:", prediction)

            print(
                result.to_string(
                    index=False,
                    formatters={
                        "Confidence": "{:.2f}%".format
                    }
                )
            )

            print("-"*70)


        except Exception as e:

            print("\nMODEL:", name)
            print("ERROR:", e)
            print("-"*70)

In [63]:
show_all_model_prediction(
     """

can't do anything in life
"""
)

Input Text:


can't do anything in life


MODEL: Naive Bayes (BOW)
Prediction: Depression
     Class Confidence
Depression     64.32%
 happiness     13.25%
      love     10.32%
      hate      3.34%
   sadness      3.27%
    Normal      2.30%
       fun      1.55%
     anger      1.26%
  surprise      0.23%
     worry      0.17%
----------------------------------------------------------------------

MODEL: Logistic Regression (BOW)
Prediction: Normal
     Class Confidence
    Normal     96.46%
Depression      2.76%
 happiness      0.21%
      love      0.17%
     anger      0.13%
   sadness      0.09%
      hate      0.09%
       fun      0.07%
     worry      0.01%
  surprise      0.01%
----------------------------------------------------------------------

MODEL: SVM (BOW)
Prediction: Normal
     Class Confidence
    Normal     57.32%
Depression      9.83%
     anger      6.02%
 happiness      4.62%
  surprise      4.26%
     worry      4.03%
   sadness      4.02%
       fun      3.